In [21]:
import pandas as pd
import numpy as np
import glob

In [22]:
for file in glob.glob("raw_data/*.parquet"):
  display(f"---File: {file} ---")
  df = pd.read_parquet(file)
  display(f"Shape: {df.shape}")
  df.info()
  #display(df.describe)
  display(df.sample(5))
  display(df.isna().sum())
  display(df.duplicated().sum())

'---File: raw_data\\distribution_centers.parquet ---'

'Shape: (10, 5)'

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        10 non-null     Int64  
 1   name                      10 non-null     str    
 2   latitude                  10 non-null     float64
 3   longitude                 10 non-null     float64
 4   distribution_center_geom  10 non-null     str    
dtypes: Int64(1), float64(2), str(2)
memory usage: 913.0 bytes


,id,name,latitude,longitude,distribution_center_geom
3,10,Savannah GA,32.0167,-81.1167,POINT(-81.1167 32.0167)
8,6,Port Authority of New York/New Jersey NY/NJ,40.6340,-73.7834,POINT(-73.7834 40.634)
2,8,Mobile AL,30.6944,-88.0431,POINT(-88.0431 30.6944)
9,2,Chicago IL,41.8369,-87.6847,POINT(-87.6847 41.8369)
1,7,Philadelphia PA,39.9500,-75.1667,POINT(-75.1667 39.95)


id                          0
name                        0
latitude                    0
longitude                   0
distribution_center_geom    0
dtype: int64

np.int64(0)

... (32 outputs left out)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 16 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   id              100000 non-null  Int64              
 1   first_name      100000 non-null  str                
 2   last_name       100000 non-null  str                
 3   email           100000 non-null  str                
 4   age             100000 non-null  Int64              
 5   gender          100000 non-null  str                
 6   state           100000 non-null  str                
 7   street_address  100000 non-null  str                
 8   postal_code     100000 non-null  str                
 9   city            100000 non-null  str                
 10  country         100000 non-null  str                
 11  latitude        100000 non-null  float64            
 12  longitude       100000 non-null  float64            
 13  traffic_source  100000 non

,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at,user_geom
89109,68048,Jason,Smith,jasonsmith@example.net,63,M,Texas,456 Randolph Haven Suite 265,76706,Robinson,United States,31.465914,-97.092392,Search,2021-07-21 04:35:00+00:00,POINT(-97.09239247 31.46591409)
35917,51371,Samuel,Daniel,samueldaniel@example.net,48,M,Guangxi Zhuang Autonomous Region,6247 Debra Forks Apt. 882,545005,Shanghai,China,24.242831,109.450505,Search,2021-10-05 13:06:00+00:00,POINT(109.4505051 24.24283112)
72749,22373,Michael,Harris,michaelharris@example.net,65,M,Queensland,59305 Anthony Shores,4807,Mulgrave,Australia,-19.933139,147.200822,Search,2022-11-09 08:20:00+00:00,POINT(147.2008221 -19.93313854)
63954,76031,Jennifer,Simpson,jennifersimpson@example.com,38,F,Nordrhein-Westfalen,2788 Brooks Extension Suite 396,41352,Korschenbroich,Germany,51.185678,6.549303,Search,2023-08-06 00:48:00+00:00,POINT(6.549303338 51.18567847)
17381,43397,James,Morris,jamesmorris@example.org,31,M,Colorado,24562 Key Alley Apt. 824,80112,Centennial,United States,39.575905,-104.859949,Search,2019-11-21 06:52:00+00:00,POINT(-104.8599487 39.57590524)


id                0
first_name        0
last_name         0
email             0
age               0
gender            0
state             0
street_address    0
postal_code       0
city              0
country           0
latitude          0
longitude         0
traffic_source    0
created_at        0
user_geom         0
dtype: int64

np.int64(0)

# SUMMARY:
**distribution_centers.parquet**  
10 rows, 5 columns, 0 missing values, 0 duplicates  
Note: Last column can be skipped - distribution_center_geom - repetition of information from columns 3 and 4.  

**events.parquet**  
2 423 896 rows, 13 columns, 1 124 674 missing values (exclusively in user_id), 0 duplicates.  
Data question: a significant portion of events are generated by anonymous sessions (user_id is missing)?  

**inventory_items.parquet**  
489 169 rows, 12 columns, 308 120 missing values (exclusively in sold_at), 0 duplicates.  
Note: Represents warehouse inventory;  
Data question: missing sold_at values indicate items currently unsold?  
Note2: The inventory_items table duplicates product attributes (category, brand, price, department, SKU, distribution center). To verify if those attributes match, and to decide if it is good decision to normalize this data  

**orders.parquet**  
124 777 rows, 9 columns, missing values in date columns (returned_at: 112 291, shipped_at: 43 686, delivered_at: 81 164), 0 duplicates.  
Data question: missing dates correspond to orders that have not reached or completed those respective fulfillment stages?  

**order_items.parquet**  
181 049 rows, 11 columns, missing values in date columns (shipped_at: 63 585, delivered_at: 118 048, returned_at: 163 075), 0 duplicates.  
Note: Individual items within orders, following a similar lifecycle pattern of missing status dates as the main orders table.  

**products.parquet**  
29 120 rows, 9 columns, few missing values (name: 2, brand: 24), 0 duplicates.  

**users.parquet**  
100 000 rows, 16 columns, 0 missing values, 0 duplicates.   
Note: Clean user base table; similar to distribution centers, the user_geom column duplicates information already stored in the latitude and longitude columns.  
Columns: first_name, last_name, email, street_address are unnecessary and will be skipped for data privacy protection.  

# Next Step: Dates validation in orders table

* **Chronological Order:** Verify `created_at <= shipped_at <= delivered_at <= returned_at`.
* **Status vs. Date Consistency:**
  * `Cancelled` / `Processing`: Only `created_at` is populated; all subsequent dates must be `NaT`.
  * `Shipped`: `created_at` and `shipped_at` are populated; `delivered_at` and `returned_at` are `NaT`.
  * `Complete`: All dates up to `delivered_at` are populated; `returned_at` is `NaT`.
  * `Returned`: All four timestamp columns are populated.

In [23]:
df = pd.read_parquet("raw_data/orders.parquet")
orders_status = df["status"].value_counts().reset_index()
orders_status.columns = ["status", "number"]
print(orders_status)

       status  number
0     Shipped   37478
1    Complete   31127
2  Processing   24809
3   Cancelled   18877
4    Returned   12486


# Order statuses with dates (expectation):  

| Status | created_at | shipped_at | delivered_at | returned_at |
| :--- | :---: | :---: | :---: | :---: |
| **Shipped** | X | X | | |
| **Complete** | X | X | X | |
| **Processing** | X | | | |
| **Cancelled** | X | | | |
| **Returned** | X | X | X | X |


In [24]:
df[["created_at","shipped_at","delivered_at", "returned_at"]].notna().groupby(df["status"]).sum()

,created_at,shipped_at,delivered_at,returned_at
status,,,,
Cancelled,18877,0,0,0
Complete,31127,31127,31127,0
Processing,24809,0,0,0
Returned,12486,12486,12486,12486
Shipped,37478,37478,0,0


In [25]:
# Shipping before creating
invalid_shipped = (df["created_at"] > df["shipped_at"]).sum()

# Deliver before shipping
invalid_delivered = (df["shipped_at"] > df["delivered_at"]).sum()

# Return before deliver
invalid_returned = (df["delivered_at"] > df["returned_at"]).sum()

print(f"Invalid (created > shipped): {invalid_shipped}")
print(f"Invalid (shipped > delivered): {invalid_delivered}")
print(f"Invalid (delivered > returned): {invalid_returned}")

Invalid (created > shipped): 0
Invalid (shipped > delivered): 0
Invalid (delivered > returned): 0


# Summary:  
Chronologic and date consistancy in `orders` table is correct